In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# If internet is OFF in settings, enable it first:
# Settings (right panel) → Internet → ON → Save → Restart session

!pip install ultralytics -q
print("✅ ultralytics installed")

In [ ]:
# Use Kaggle's pre-built ultralytics dataset instead of pip
import subprocess
result = subprocess.run(
    ["pip", "install", "ultralytics", "--no-index",
     "--find-links", "/kaggle/input"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

import os, shutil, yaml
from pathlib import Path

BASE = "/kaggle/input/datasets/ashishverma10/edge-dataset"   # ← fixed
OUTPUT = "/kaggle/working/merged_dataset"

for split in ["images/train", "images/val", "labels/train", "labels/val"]:
    Path(f"{OUTPUT}/{split}").mkdir(parents=True, exist_ok=True)

print("✅ Dirs ready")
print("Datasets found:", os.listdir(BASE))

In [ ]:
import os

def explore(path, depth=0, max_depth=4):
    if depth > max_depth:
        return
    try:
        items = os.listdir(path)
        for item in sorted(items):
            full = os.path.join(path, item)
            indent = "  " * depth
            if os.path.isdir(full):
                print(f"{indent}📁 {item}/")
                explore(full, depth+1, max_depth)
            else:
                print(f"{indent}📄 {item}")
    except PermissionError:
        pass

print("=== /kaggle/input ===")
explore("/kaggle/input", max_depth=3)

In [ ]:
# ============================================================
# UNIFIED 4-CLASS SCHEMA
#   0 = motorcycle  |  1 = person
#   2 = helmet      |  3 = no_helmet
# ============================================================

REMAP = {
    "rider-helmet-yosia": {
        0: 2,    # helm       → helmet
        1: 1,    # pejalan    → person
        2: 0,    # pemotor    → motorcycle
        3: 3,    # tanpa-helm → no_helmet
    },
    "bike-helmet-santhosh": {
        0: 2,    # With Helmet    → helmet
        1: 3,    # Without Helmet → no_helmet
    },
    "helmet-bike-wangbo": {
        0: 0,    # bike           → motorcycle
        1: 0,    # electric_bike  → motorcycle
        2: 2,    # with_helmet    → helmet
        3: 3,    # without_helmet → no_helmet
    },
    "helmet-rider-cs174": {
        0: 2,    # full-faced    → helmet
        1: 2,    # half-faced    → helmet
        2: 1,    # helmetRider   → person
        3: None, # invalid       → SKIP
        4: 3,    # no-helmet     → no_helmet
        5: 1,    # noHelmetRider → person
    },
}

print("✅ Remap config ready")

In [ ]:
SEED = 42
VAL_RATIO = 0.15
random.seed(SEED)

CLASS_NAMES = {0: "motorcycle", 1: "person", 2: "helmet", 3: "no_helmet"}
label_counts = {0: 0, 1: 0, 2: 0, 3: 0}
total_kept = 0
total_dropped = 0

def remap_and_write(src_lbl, dst_lbl, mapping):
    """Remap one label file. Returns True if file had valid labels."""
    new_lines = []
    for line in Path(src_lbl).read_text().strip().splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        old_cls = int(parts[0])
        new_cls = mapping.get(old_cls)
        if new_cls is None:
            continue
        new_lines.append(f"{new_cls} {' '.join(parts[1:])}")
        label_counts[new_cls] += 1
    if new_lines:
        Path(dst_lbl).write_text("\n".join(new_lines))
        return True
    return False

all_samples = []  # (img_path, lbl_path, ds_name)

for ds_name in REMAP:
    ds_path = Path(BASE) / ds_name
    mapping = REMAP[ds_name]

    # Find all split folders (train/valid/val/test)
    for split_dir in ds_path.iterdir():
        if not split_dir.is_dir():
            continue
        img_dir = split_dir / "images"
        lbl_dir = split_dir / "labels"
        if not img_dir.exists() or not lbl_dir.exists():
            # Try reversed structure: images/train, labels/train
            img_dir = ds_path / "images" / split_dir.name
            lbl_dir = ds_path / "labels" / split_dir.name

        if img_dir.exists() and lbl_dir.exists():
            for img in img_dir.glob("*.*"):
                lbl = lbl_dir / (img.stem + ".txt")
                if lbl.exists():
                    all_samples.append((img, lbl, ds_name))

print(f"📊 Total samples found: {len(all_samples)}")

# Fresh train/val split across all datasets
random.shuffle(all_samples)
n_val = int(len(all_samples) * VAL_RATIO)
splits_map = {
    "val":   all_samples[:n_val],
    "train": all_samples[n_val:]
}

for split_name, samples in splits_map.items():
    kept = dropped = 0
    for img_path, lbl_path, ds_name in samples:
        mapping = REMAP[ds_name]
        dst_img = Path(OUTPUT) / "images" / split_name / img_path.name
        dst_lbl = Path(OUTPUT) / "labels" / split_name / (img_path.stem + ".txt")

        ok = remap_and_write(lbl_path, dst_lbl, mapping)
        if ok:
            shutil.copy(img_path, dst_img)
            kept += 1
        else:
            dropped += 1
        total_kept += kept if split_name == "train" else 0

    print(f"  {split_name}: ✅ {kept} kept, ⚠️ {dropped} dropped (no valid labels)")

print(f"\n📊 Label distribution (train):")
for cls_id, count in label_counts.items():
    print(f"  {cls_id} ({CLASS_NAMES[cls_id]}): {count} instances")

In [ ]:
yaml_path = f"{OUTPUT}/data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump({
        "path":  OUTPUT,
        "train": "images/train",
        "val":   "images/val",
        "nc":    4,
        "names": ["motorcycle", "person", "helmet", "no_helmet"]
    }, f)

print("✅ data.yaml:")
print(open(yaml_path).read())

In [ ]:
import cv2
import matplotlib.pyplot as plt

COLORS = {0:(255,140,0), 1:(0,200,255), 2:(0,220,100), 3:(255,50,50)}

def draw_boxes(img_path, lbl_path):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for line in Path(lbl_path).read_text().strip().splitlines():
        p = line.split()
        cls = int(p[0])
        cx,cy,bw,bh = float(p[1]),float(p[2]),float(p[3]),float(p[4])
        x1,y1 = int((cx-bw/2)*w), int((cy-bh/2)*h)
        x2,y2 = int((cx+bw/2)*w), int((cy+bh/2)*h)
        c = COLORS.get(cls,(200,200,200))
        cv2.rectangle(img,(x1,y1),(x2,y2),c,2)
        cv2.putText(img,CLASS_NAMES[cls],(x1,y1-6),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,c,2)
    return img

train_imgs = list((Path(OUTPUT)/"images/train").glob("*.*"))
random.shuffle(train_imgs)

fig, axes = plt.subplots(2, 3, figsize=(15,8))
for ax, img_path in zip(axes.flatten(), train_imgs[:6]):
    lbl = Path(OUTPUT)/"labels/train"/(img_path.stem+".txt")
    if lbl.exists():
        ax.imshow(draw_boxes(img_path, lbl))
        ax.set_title(img_path.stem[:20], fontsize=8)
    ax.axis("off")
plt.suptitle("Merged Dataset — Remapped Labels Check")
plt.tight_layout()
plt.show()

In [ ]:
from ultralytics import YOLO

# Start fresh or resume from checkpoint
RESUME = False  # ← set True to resume from last checkpoint

model = YOLO("/kaggle/working/helmet_v1/weights/last.pt" if RESUME else "yolov8n.pt")

model.train(
    data=yaml_path,
    epochs=30,           # ← start with 30, check results, then extend
    imgsz=320,
    batch=32,
    device=0,
    workers=4,
    mosaic=1.0,
    mixup=0.1,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    lr0=0.01,
    lrf=0.01,
    warmup_epochs=3,
    weight_decay=0.0005,
    project="/kaggle/working",
    name="helmet_v1",
    save=True,
    save_period=10,      # ← checkpoint every 10 epochs (so you get epoch10, 20, 30)
    resume=RESUME,
    plots=True,
)

In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/helmet_v1/weights/last.pt")

model.train(
    data=yaml_path,
    epochs=50,              # 50 more → total ~80
    imgsz=640,              # ← KEY: 320→640 will boost small object detection
    batch=16,               # reduced from 32 (640 needs more VRAM)
    device=0,
    workers=4,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,         # new: helps helmet instances
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    lr0=0.001,              # lower LR for fine-tuning continuation
    lrf=0.01,
    weight_decay=0.0005,
    project="/kaggle/working",
    name="helmet_v2",       # new name → won't overwrite v1
    save=True,
    save_period=10,
    resume=False,           # False because we're changing imgsz (can't resume with diff imgsz)
    plots=True,
)

In [ ]:
from ultralytics import YOLO

yaml_path = "/kaggle/working/merged_dataset/data.yaml"

# Check saved_weights
m1 = YOLO("/kaggle/working/saved_weights/best.pt")
r1 = m1.val(data=yaml_path, imgsz=640, verbose=False)
print(f"saved_weights/best.pt mAP50: {r1.box.map50:.4f}")

# Check helmet_v2
m2 = YOLO("/kaggle/working/helmet_v2/weights/best.pt")
r2 = m2.val(data=yaml_path, imgsz=640, verbose=False)
print(f"helmet_v2/best.pt mAP50: {r2.box.map50:.4f}")

In [ ]:
model = YOLO("/kaggle/working/saved_weights/best.pt")

model.export(
    format="ncnn",
    imgsz=640,
    half=True,
)

NCNN FP16 Accuracy

In [ ]:
model_ncnn = YOLO("/kaggle/working/saved_weights/best_ncnn_model")
metrics = model_ncnn.val(data=yaml_path, imgsz=640, half=True)
print(f"NCNN FP16 mAP50: {metrics.box.map50:.4f}")
print(f"Accuracy drop: {0.8165 - metrics.box.map50:.4f}")

In [ ]:
import shutil
from IPython.display import FileLink

# Zip NCNN model
shutil.make_archive("/kaggle/working/saved_weights/best_ncnn", "zip", 
                    "/kaggle/working/saved_weights/best_ncnn_model")

display(FileLink("/kaggle/working/saved_weights/best.pt"))
display(FileLink("/kaggle/working/saved_weights/best_ncnn.zip"))

In [ ]:
import glob
images = glob.glob("/kaggle/working/merged_dataset/images/val/*.jpg")
print(f"Total val images: {len(images)}")
print(images[:5])  # see sample paths

In [ ]:
from ultralytics import YOLO
import cv2, glob, os

model = YOLO("/kaggle/working/saved_weights/best_ncnn_model")

def box_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-6)

images = sorted(glob.glob("/kaggle/working/merged_dataset/images/val/*.jpg"))[:100]

os.makedirs("/kaggle/working/frame_output", exist_ok=True)

for idx, img_path in enumerate(images):
    frame = cv2.imread(img_path)
    frame = cv2.resize(frame, (640, 480))  # uniform size
    results = model(frame, imgsz=640, half=True, verbose=False)
    annotated = results[0].plot()

    for r in results:
        boxes   = r.boxes.xyxy.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy()
        motorcycles = boxes[classes == 0]
        persons     = boxes[classes == 1]
        no_helmets  = boxes[classes == 3]

        for i, moto in enumerate(motorcycles):
            riders     = [p for p in persons   if box_iou(moto, p) > 0.1]
            violations = [n for n in no_helmets if box_iou(moto, n) > 0.1]
            if len(riders) >= 3:
                cv2.putText(annotated, "TRIPLE RIDING!", (20, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,0,255), 3)
            if violations:
                cv2.putText(annotated, "HELMET VIOLATION!", (20, 100),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,0,255), 3)

    cv2.imwrite(f"/kaggle/working/frame_output/frame_{idx:03d}.jpg", annotated)
    if idx % 10 == 0:
        print(f"Processed {idx}/100")

# Stitch into video
frames = sorted(glob.glob("/kaggle/working/frame_output/*.jpg"))
out = cv2.VideoWriter("/kaggle/working/detection_output.mp4",
                      cv2.VideoWriter_fourcc(*"mp4v"), 3, (640, 480))
for f in frames:
    out.write(cv2.imread(f))
out.release()

print("✅ Done!")

from IPython.display import FileLink
display(FileLink("/kaggle/working/detection_output.mp4"))

In [ ]:
import glob

# Check what your val images look like — pick best ones
images = sorted(glob.glob("/kaggle/working/merged_dataset/images/val/*.jpg"))

# Run detection on 10 random images and save
import cv2, random
from ultralytics import YOLO

model = YOLO("/kaggle/working/saved_weights/best_ncnn_model")

samples = random.sample(images, 10)

os.makedirs("/kaggle/working/test_samples", exist_ok=True)

for idx, img_path in enumerate(samples):
    results = model(img_path, imgsz=640, half=True, verbose=False)
    annotated = results[0].plot()
    out_path = f"/kaggle/working/test_samples/result_{idx}.jpg"
    cv2.imwrite(out_path, annotated)
    print(f"Saved: {out_path}")

from IPython.display import FileLink, Image
# Show first result inline
display(Image(filename="/kaggle/working/test_samples/result_0.jpg"))

summary

In [ ]:
from ultralytics import YOLO
import cv2, glob, os, random
from datetime import datetime

model = YOLO("/kaggle/working/saved_weights/best_ncnn_model")

def box_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-6)

def analyze_image(img_path, save_path=None):
    results = model(img_path, imgsz=640, half=True, conf=0.25, verbose=False)
    annotated = results[0].plot()

    report = {
        "image"      : os.path.basename(img_path),
        "total_bikes": 0,
        "bikes"      : []
    }

    for r in results:
        boxes   = r.boxes.xyxy.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy()

        motorcycles = boxes[classes == 0]
        helmets     = boxes[classes == 2]
        no_helmets  = boxes[classes == 3]

        report["total_bikes"] = len(motorcycles)

        for i, moto in enumerate(motorcycles):
            helm_on    = [h for h in helmets    if box_iou(moto, h) > 0.01]
            no_helm_on = [n for n in no_helmets if box_iou(moto, n) > 0.01]

            has_violation = len(no_helm_on) > 0

            bike_info = {
                "bike_id"         : i + 1,
                "helmet_worn"     : len(helm_on),
                "no_helmet"       : len(no_helm_on),
                "helmet_violation": has_violation,
            }
            report["bikes"].append(bike_info)

            # Annotate box
            x1, y1, x2, y2 = moto.astype(int)
            color = (0, 0, 255) if has_violation else (0, 255, 0)
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            label = f"Bike {i+1} | {'NO HELMET!' if has_violation else 'SAFE'}"
            cv2.putText(annotated, label, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    if save_path:
        cv2.imwrite(save_path, annotated)

    return report

# ── Run ──────────────────────────────────────────────────
all_images = sorted(glob.glob("/kaggle/working/merged_dataset/images/val/*.jpg"))
samples    = random.sample(all_images, 20)
os.makedirs("/kaggle/working/test_samples", exist_ok=True)

grand_bikes     = 0
grand_violation = 0
grand_safe      = 0

print("=" * 55)
print("         PER IMAGE DETECTION REPORT")
print("=" * 55)

for idx, img_path in enumerate(samples):
    save_path = f"/kaggle/working/test_samples/result_{idx:03d}.jpg"
    report    = analyze_image(img_path, save_path)

    print(f"\n📷 Image {idx+1}: {report['image']}")
    print(f"   Total Bikes : {report['total_bikes']}")

    for bike in report["bikes"]:
        status = "🚨 NO HELMET" if bike["helmet_violation"] else "✅ SAFE"
        print(f"   Bike {bike['bike_id']}: "
              f"Helmet={bike['helmet_worn']} | "
              f"NoHelmet={bike['no_helmet']} | {status}")

        grand_bikes     += 1
        grand_violation += 1 if bike["helmet_violation"] else 0
        grand_safe      += 1 if not bike["helmet_violation"] else 0

print("\n" + "=" * 55)
print("           GRAND SUMMARY REPORT")
print(f"           {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 55)
print(f"  Images Analyzed    : {len(samples)}")
print(f"  Total Bikes        : {grand_bikes}")
print(f"  🚨 Helmet Violations: {grand_violation}")
print(f"  ✅ Safe Bikes       : {grand_safe}")
if grand_bikes > 0:
    print(f"  Violation Rate     : {(grand_violation/grand_bikes)*100:.1f}%")
print("=" * 55)

In [ ]:
from ultralytics import YOLO
import cv2
from IPython.display import display, Image as IPImage

model = YOLO("/kaggle/working/saved_weights/best_ncnn_model")

def box_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-6)

def test_image(img_path):
    results = model(img_path, imgsz=640, half=True, conf=0.25, verbose=False)
    annotated = results[0].plot()

    for r in results:
        boxes   = r.boxes.xyxy.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy()

        motorcycles = boxes[classes == 0]
        helmets     = boxes[classes == 2]
        no_helmets  = boxes[classes == 3]

        print(f"Total Bikes : {len(motorcycles)}")
        for i, moto in enumerate(motorcycles):
            helm_on    = [h for h in helmets    if box_iou(moto, h) > 0.01]
            no_helm_on = [n for n in no_helmets if box_iou(moto, n) > 0.01]
            status     = "🚨 NO HELMET" if no_helm_on else "✅ SAFE"
            print(f"  Bike {i+1}: Helmet={len(helm_on)} | NoHelmet={len(no_helm_on)} | {status}")

    # Save and display
    out_path = "/kaggle/working/test_result.jpg"
    cv2.imwrite(out_path, annotated)
    display(IPImage(filename=out_path))

# ← change path to your image
test_image("/kaggle/input/datasets/ashishverma10/tripple-test/triipple.png")

In [ ]:
import os
file_exists = os.path.exists("/kaggle/input/datasets/ashishverma10/tripple-test/triipple.png")
print("File exists:", file_exists)

In [ ]:
from ultralytics import YOLO
import cv2
import glob
import random
from datetime import datetime
from IPython.display import display, Image as IPImage

# ── Config ───────────────────────────────────────────────
MODEL_PATH = "/kaggle/working/saved_weights/best_ncnn_model"
IMGSZ      = 640
CONF       = 0.25

model = YOLO(MODEL_PATH, task="detect")

def box_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-6)

def test_single_image(img_path):
    results   = model(img_path, imgsz=IMGSZ, half=True, conf=CONF, verbose=False)
    annotated = results[0].plot()

    print(f"\n📷 {img_path.split('/')[-1]}")
    print("-" * 45)

    for r in results:
        boxes   = r.boxes.xyxy.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy()

        motorcycles = boxes[classes == 0]
        helmets     = boxes[classes == 2]
        no_helmets  = boxes[classes == 3]

        print(f"Total Bikes : {len(motorcycles)}")

        for i, moto in enumerate(motorcycles):
            helm_on    = [h for h in helmets    if box_iou(moto, h) > 0.01]
            no_helm_on = [n for n in no_helmets if box_iou(moto, n) > 0.01]

            # Riders estimated from head detections
            estimated_riders = len(helm_on) + len(no_helm_on)
            has_violation    = len(no_helm_on) > 0

            # Annotate
            x1, y1, x2, y2 = moto.astype(int)
            color = (0, 0, 255) if has_violation else (0, 255, 0)
            cv2.rectangle(annotated, (x1,y1), (x2,y2), color, 2)
            label = f"Bike{i+1} R:{estimated_riders} {'NO HELMET!' if has_violation else 'SAFE'}"
            cv2.putText(annotated, label, (x1, max(y1-10,10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            print(f"  Bike {i+1}: Riders~{estimated_riders} | "
                  f"Helmet={len(helm_on)} | "
                  f"NoHelmet={len(no_helm_on)} | "
                  f"{'🚨 VIOLATION' if has_violation else '✅ SAFE'}")

    out = "/kaggle/working/result.jpg"
    cv2.imwrite(out, annotated)
    display(IPImage(filename=out))

# ── Test ─────────────────────────────────────────────────
# Val images
images  = glob.glob("/kaggle/working/merged_dataset/images/val/*.jpg")
samples = random.sample(images, 5)
for img in samples:
    test_single_image(img)

# Your custom image
test_single_image("/kaggle/input/datasets/ashishverma10/tripple-test/triipple.png")

In [ ]:
from ultralytics import YOLO
import cv2
from datetime import datetime
from IPython.display import display, Image as IPImage

MODEL_PATH = "/kaggle/working/saved_weights/best_ncnn_model"
IMAGE_PATH = "/kaggle/input/datasets/ashishverma10/tripple-test/triipple.png"
IMGSZ      = 640
CONF       = 0.25

model = YOLO(MODEL_PATH, task="detect")

def box_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter  = max(0, x2-x1) * max(0, y2-y1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-6)

# ── Inference ────────────────────────────────────────────
results   = model(IMAGE_PATH, imgsz=IMGSZ, half=True, conf=CONF, verbose=False)
annotated = results[0].plot()

total_bikes       = 0
helmet_violations = 0
triple_violations = 0
safe_bikes        = 0

for r in results:
    boxes   = r.boxes.xyxy.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy()

    motorcycles = boxes[classes == 0]
    helmets     = boxes[classes == 2]
    no_helmets  = boxes[classes == 3]

    total_bikes = len(motorcycles)

    for i, moto in enumerate(motorcycles):
        # Step 1: count overlapping helmets/no_helmets
        helm_on    = [h for h in helmets    if box_iou(moto, h) > 0.01]
        no_helm_on = [n for n in no_helmets if box_iou(moto, n) > 0.01]

        # Step 2: estimate riders
        estimated_riders = len(helm_on) + len(no_helm_on)

        # Step 3: check violations
        is_triple     = estimated_riders > 2
        has_violation = len(no_helm_on) > 0

        if is_triple:                          triple_violations += 1
        if has_violation:                      helmet_violations += 1
        if not is_triple and not has_violation: safe_bikes       += 1

        # Step 4: annotate
        x1, y1, x2, y2 = moto.astype(int)
        color = (0, 0, 255) if (has_violation or is_triple) else (0, 255, 0)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)

        tags = []
        if is_triple    : tags.append("TRIPLE!")
        if has_violation: tags.append("NO HELMET!")
        if not tags     : tags.append("SAFE")

        label = f"Bike{i+1} R:{estimated_riders} | {' | '.join(tags)}"
        cv2.putText(annotated, label, (x1, max(y1-10, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

# ── Save & Display ───────────────────────────────────────
out = "/kaggle/working/result.jpg"
cv2.imwrite(out, annotated)
display(IPImage(filename=out))

# ── Summary Report ───────────────────────────────────────
print("=" * 45)
print("       TRAFFIC VIOLATION REPORT")
print(f"       {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 45)
print(f"  Total Bikes         : {total_bikes}")
print(f"  🚨 Helmet Violations : {helmet_violations}")
print(f"  🚨 Triple Riding     : {triple_violations}")
print(f"  ✅ Safe Bikes        : {safe_bikes}")
print("=" * 45)

In [ ]:
for r in results:
    boxes   = r.boxes.xyxy.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy()
    confs   = r.boxes.conf.cpu().numpy()
    
    names = {0:"motorcycle", 1:"person", 2:"helmet", 3:"no_helmet"}
    for box, cls, conf in zip(boxes, classes, confs):
        print(f"{names[int(cls)]:12s} conf={conf:.2f} box={box.astype(int)}")